# Excercise 2 - Fairness metrics

This notebook demonstrates fairness evaluation and mitigation techniques.

1. Train a baseline logistic regression and evaluate predictive performance.
2. Measure fairness using demographic parity and equalized odds.
3. Mitigate unfairness (1): post-process predictions with `ThresholdOptimizer`
3. Mitigate unfairness (2): train with `ExponentiatedGradient`
4. Compare the results of the two mitigation techniques and discuss trade-offs.


In [ ]:
import json

import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.compose import make_column_selector as selector
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score

from fairlearn.datasets import fetch_adult
from fairlearn.postprocessing import ThresholdOptimizer
from fairlearn.metrics import demographic_parity_ratio, equalized_odds_ratio
from fairlearn.reductions import DemographicParity, ExponentiatedGradient

def get_class_metrics(y_true, y_pred):
    """Compute common classification metrics, print them, and return as a dict."""
    scores = {
        "roc_auc": roc_auc_score(y_true, y_pred),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred),
        "recall": recall_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred),
    }
    print(f"ROC-AUC: {scores['roc_auc']}")
    print(f"Accuracy: {scores['accuracy']}")
    print(f"Precision: {scores['precision']}")
    print(f"Recall: {scores['recall']}")
    print(f"F1: {scores['f1']}")
    return scores

## Fetching the data and setup the training and test datasets

In [ ]:
# Source: https://archive.ics.uci.edu/dataset/2/adult
data = fetch_adult(as_frame=True)

target_labels = (data.target == ">50K") * 1
df = data.data.copy()
sensitive_features = df["sex"]

(X_train, X_test, y_train, y_test, A_train, A_test) = train_test_split(
    df, target_labels, sensitive_features, test_size=0.3, random_state=42, stratify=target_labels
)

X_train = X_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)
A_train = A_train.reset_index(drop=True)
A_test = A_test.reset_index(drop=True)

plot_df = df.copy()
plot_df["target"] = data.target

In [ ]:
df.head()

## Simple stats

In [ ]:
plot_df["sex"].value_counts(normalize = True)

In [ ]:
plot_df.groupby("sex")["target"].value_counts(normalize = True) * 100

## Creating the pipeline for the logistic regression model

In [ ]:
# Numeric feature processing: impute missing values, then scale
numeric_transformer = Pipeline(
    steps=[
        ("impute", SimpleImputer()),  # fill missing numeric values (default: mean)
        ("scaler", StandardScaler()),  # standardize to zero mean and unit variance
    ]
)
# Categorical feature processing: impute most frequent, then one-hot encode
categorical_transformer = Pipeline(
    [
        ("impute", SimpleImputer(strategy="most_frequent")),  # fill missing with mode
        ("ohe", OneHotEncoder(handle_unknown="ignore")),  # convert categories to binary features
    ]
)
# Combine numeric and categorical pipelines using selectors
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, selector(dtype_exclude="category")),  # apply to numeric-like columns
        ("cat", categorical_transformer, selector(dtype_include="category")),  # apply to categorical columns
    ]
)

# Full modeling pipeline: preprocessing -> classifier
pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),  # transforms raw features into numeric array
        (
            "classifier",
            LogisticRegression(solver="liblinear", fit_intercept=True),
        ),
    ]
)

## Running the baseline model

In [ ]:
# To keep track of the metrics for each model, we will store them in a dictionary.
metrics = {}

In [ ]:
pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)

metrics["baseline"] = get_class_metrics(y_test, y_pred)

## Calculating the demographic parity ratio and the equal odds ratio

 - **Demographic parity ratio** - Shows the ratio of the lowest and highest selection rates, so a result of 1 means there is demographic parity.
 - **Equal odds ratio** - The equalized odds ratio of 1 means that all groups have the same true positive, true negative, false positive, and false negative rates.

For the baseline model both metrics is low, due to the unfairness between the genders.

In [ ]:
metrics["baseline"]["demographic_parity"] = demographic_parity_ratio(y_test, y_pred, sensitive_features=A_test)
metrics["baseline"]["equalized_odds"] = equalized_odds_ratio(y_test, y_pred, sensitive_features=A_test)


print(f'Value of demographic parity ratio: {round(metrics["baseline"]["demographic_parity"], 3)}')
print(f'Value of equal odds ratio: {round(metrics["baseline"]["equalized_odds"], 3)}')

## Reducing unfairness with Threshold Optimizer

Apply model post-processing using `ThresholdOptimizer` from Fairlearn. This post-processor learns group-specific decision thresholds to map model scores (probabilities) to binary predictions so that a chosen fairness constraint (for example, demographic parity) is better satisfied on the training set.

How it works:
- `ThresholdOptimizer` takes a probabilistic estimator (an estimator exposing `predict_proba`) and, for each sensitive group, finds a threshold on the predicted probability that optimizes the selected fairness constraint while trading off utility.
- During `fit`, you provide `sensitive_features` so the optimizer can learn separate thresholds per group. During `predict`, pass the same `sensitive_features` to apply the learned thresholds.

Key parameters and usage notes:
- `predict_method='predict_proba'`: use model probabilities as inputs for thresholding.
- `constraints='demographic_parity'` (or others): selects the fairness objective to enforce.

Pros and cons:
- Pros: model-agnostic post-processing, simple to apply, and works with any probabilistic estimator.
- Cons: it modifies final decisions (not the model internals), may reduce overall accuracy, and requires sufficient representation of each sensitive group in the training data to learn reliable thresholds.

After fitting the optimizer we inspect the learned thresholds and evaluate fairness and performance metrics on the test set to understand trade-offs introduced by the post-processing step.

In [ ]:
threshold_optimizer = ThresholdOptimizer(
    estimator=pipeline,
    constraints="demographic_parity",
    predict_method="predict_proba",
    prefit=False,
)

threshold_optimizer.fit(X_train, y_train, sensitive_features=A_train)

y_pred_opt = threshold_optimizer.predict(X_test, sensitive_features=A_test)

print(
    json.dumps(
        threshold_optimizer.interpolated_thresholder_.interpolation_dict,
        default=str,
        indent=4,
    )
)

The printed JSON shows the per-group decision rules learned by the `ThresholdOptimizer`'s internal interpolated thresholder. Each top-level key is a sensitive-group label (for example, `Female` or `Male`). For each group you will typically see one or two operations with associated mixing probabilities:

- `operation0`, `operation1`: string conditions of the form `[>threshold]` that are evaluated against the model's predicted probability (e.g., `predict_proba` score for the positive class).
- If `operation0` is true, the post-processor returns the positive label for that invocation.
- If `operation0` is false but `operation1` is true, it returns the positive label with probability `p1`.
- If both `operation0` and `operation1` are false, it returns the negative label.


Interpretation:
- For a female sample if the predicted probability exceeds 0.1892, the post-processor will return the positive label; if it is between 0.1833 and 0.1892, it will return the positive label with probability ~0.156; if it is below 0.1833, it will return the negative label.
- For a male sample if the predicted probability exceeds 0.694, the post-processor will return the positive label; if it is between 0.662 and 0.694, it will return the positive label with probability ~0.846; if it is below 0.662, it will return the negative label.


After the threshold optimization, the model evaluation metrics shows similar values and the demographic parity ratio significantly increased.

In [ ]:
metrics["post-processed"] = get_class_metrics(y_test, y_pred_opt)
metrics["post-processed"]["demographic_parity"] = demographic_parity_ratio(y_test, y_pred_opt, sensitive_features=A_test)
metrics["post-processed"]["equalized_odds"] = equalized_odds_ratio(y_test, y_pred_opt, sensitive_features=A_test)


print(f'Value of demographic parity ratio (after post-processing): {round(metrics["post-processed"]["demographic_parity"], 2)}')
print(f'Value of equal odds ratio (after post-processing): {round(metrics["post-processed"]["equalized_odds"], 2)}')

## Reducing unfairness with Exponentiated Gradient

`ExponentiatedGradient` is an in-processing algorithm that enforces a chosen fairness constraint by reducing the problem to a sequence of cost-sensitive classification problems.

How it works (high level): a reduction technique to mitigate model bias by treating fairness constraints as an optimization problem, generating a sequence of re-weighted datasets.

Key parameters and notes:
- `constraints`: the fairness objective (e.g., `DemographicParity()`).
- `eps`: tolerance for fairness violation (smaller values enforce fairness more strictly but may increase iterations).
- `sample_weight_name`: name used to pass sample weights into the base estimator when required.

Pros and cons:
- Pros: finds classifiers that directly optimize for the specified fairness constraint and utility trade-off; model-agnostic (works with any compatible base learner).
- Cons: can be slower than post-processing (requires multiple trainings of the base learner), and it may produce a randomized mixture of classifiers which can complicate deployment if deterministic behavior is required.

Usage tip: use `ExponentiatedGradient` when you want to learn a model that incorporates fairness during training (rather than adjusting predictions afterwards). If you need deterministic single-model outputs, consider averaging predictions or selecting the best single classifier from the mixture, while checking fairness consequences.

In [ ]:
exponentiated_gradient = ExponentiatedGradient(
    estimator=pipeline,
    constraints=DemographicParity(),
    sample_weight_name="classifier__sample_weight",
)
exponentiated_gradient.fit(X_train, y_train, sensitive_features=A_train)
y_pred_exp = exponentiated_gradient.predict(X_test)

In [ ]:
metrics["exponentiated-gradient"] = get_class_metrics(y_test, y_pred_exp)
metrics["exponentiated-gradient"]["demographic_parity"] = demographic_parity_ratio(y_test, y_pred_exp, sensitive_features=A_test)
metrics["exponentiated-gradient"]["equalized_odds"] = equalized_odds_ratio(y_test, y_pred_exp, sensitive_features=A_test)


print(f'Value of demographic parity ratio (after post-processing): {round(metrics["exponentiated-gradient"]["demographic_parity"], 2)}')
print(f'Value of equal odds ratio (after post-processing): {round(metrics["exponentiated-gradient"]["equalized_odds"], 2)}')

## Comparing the results of the two mitigation techniques

In [ ]:
pd.DataFrame(metrics).T